# Adaptive Reward Checkpoint With Fresh Trade Restart

This notebook builds an `adaptive_reward` snapshot using long-history data, then resumes trading from that snapshot without carrying over prior positions, cash, or trade logs.

What is preserved:
- Daily probability model
- 5-minute buy/sell models
- Adaptive-reward threshold history
- Chan warmup bars and feature state needed to continue training

What is reset on resume:
- Execution engine state
- Open position
- Trade log
- Equity path


In [ ]:
from pathlib import Path
import pandas as pd

from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot,
)
from adaptive_trade_extensions import RollingThresholdConfig, make_threshold_grid

snapshot_path = Path("checkpoints") / "SPY_adaptive_reward_fresh_start.joblib"

common_kwargs = dict(
    daily_csv_path="DataAPI/data/SPY_DAY.csv",
    k5m_csv_path="DataAPI/data/SPY_5M.csv",
    code="SPY",
    daily_chan_start="2008-01-01",
    accumulation_start="2010-01-01",
    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=5,
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={
        "vix_": "VIX.csv",
    },
    static_buy_level=0.20,
    static_sell_level=0.30,
    daily_threshold_config=RollingThresholdConfig(
        lookback_days=15,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    verbose=True,
)


## 1. Build And Save The Snapshot

This step trains using daily history starting in 2008 and 5-minute history starting in 2010, then saves a reusable snapshot.

In [8]:
snapshot_res = build_adaptive_reward_snapshot(
    snapshot_path=str(snapshot_path),
    snapshot_end_time="2023-12-31",
    output_dir="output_adaptive_reward_snapshot_build_SPY",
    **common_kwargs,
)

print(snapshot_res["snapshot_path"])
display(snapshot_res["daily_reward_df"].tail())
display(snapshot_res["daily_log_df"].tail())


[TRAIN][DAILY-PROB] n=200 pos=43 (21.50%)
[TRAIN][DAILY-PROB] n=225 pos=49 (21.78%)
[TRAIN][DAILY-PROB] n=250 pos=57 (22.80%)
[TRAIN][DAILY-PROB] n=275 pos=66 (24.00%)
[TRAIN][DAILY-PROB] n=300 pos=74 (24.67%)
[TRAIN][DAILY-PROB] n=325 pos=78 (24.00%)
[TRAIN][DAILY-PROB] n=350 pos=85 (24.29%)
[TRAIN][DAILY-PROB] n=375 pos=94 (25.07%)
[TRAIN][DAILY-PROB] n=400 pos=101 (25.25%)
[TRAIN][DAILY-PROB] n=425 pos=104 (24.47%)
[TRAIN][DAILY-PROB] n=450 pos=112 (24.89%)
[TRAIN][DAILY-PROB] n=475 pos=112 (23.58%)
[TRAIN][DAILY-PROB] n=500 pos=113 (22.60%)
[TRAIN][DAILY-PROB] n=525 pos=124 (23.62%)
[TRAIN][DAILY-PROB] n=550 pos=131 (23.82%)
[TRAIN][DAILY-PROB] n=575 pos=143 (24.87%)
[TRAIN][DAILY-PROB] n=600 pos=150 (25.00%)
[TRAIN][DAILY-PROB] n=625 pos=150 (24.00%)
[TRAIN][DAILY-PROB] n=650 pos=158 (24.31%)
[TRAIN][DAILY-PROB] n=675 pos=160 (23.70%)
[TRAIN][DAILY-PROB] n=700 pos=161 (23.00%)
[TRAIN][DAILY-PROB] n=725 pos=168 (23.17%)
[TRAIN][DAILY-PROB] n=750 pos=173 (23.07%)
[TRAIN][DAILY-PROB]

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
3517,2023-12-22,0.150978,0.2,0.3,FORCE_BUY,-0.003475,-0.002643,0.000000,-0.003475,FORCE_SELL,2.536816e+09,473.18,1.95,0.75
3518,2023-12-26,0.136758,0.2,0.3,FORCE_BUY,0.002487,0.002162,0.000190,0.002487,FORCE_BUY,2.543126e+09,475.59,1.55,-0.50
3519,2023-12-27,0.114461,0.2,0.3,FORCE_BUY,0.002649,0.004244,0.000105,0.002649,FREE,2.553920e+09,476.90,1.55,-0.50
3520,2023-12-28,0.139579,0.2,0.3,FORCE_BUY,0.000000,0.000189,-0.000210,0.000000,FREE,2.554402e+09,477.05,-0.50,1.40
3521,2023-12-29,0.154243,0.2,0.3,FORCE_BUY,-0.004881,-0.005516,0.000126,-0.004881,FORCE_SELL,2.554723e+09,475.07,1.80,1.40


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
3517,2023-12-22,795220.002406,0.0,1,1.55,-0.5,0.150978,FORCE_BUY,0.2,0.3
3518,2023-12-26,799270.216290,0.0,1,1.55,-0.5,0.136758,FORCE_BUY,0.2,0.3
3519,2023-12-27,801471.784833,0.0,1,-0.50,1.4,0.114461,FORCE_BUY,0.2,0.3
3520,2023-12-28,801723.872834,0.0,1,1.80,1.4,0.139579,FORCE_BUY,0.2,0.3
3521,2023-12-29,798396.311220,0.0,1,1.80,1.4,0.154243,FORCE_BUY,0.2,0.3


## 2. Resume With Fresh Trading State

This step loads the saved models and adaptive-reward history, but it starts trading fresh from the next period using a brand-new execution engine.

In [9]:
resume_res = run_adaptive_reward_from_snapshot(
    snapshot_path=str(snapshot_path),
    end_time="2026-12-31",
    sim_start="2024-01-01",
    initial_capital=100000.0,
    fee_pct=0.0,
    output_dir="output_adaptive_reward_resumed_fresh_SPY_15day",
    # Optional. If omitted, the code saves next to the source snapshot as
    # checkpoints/SPY_adaptive_reward_fresh_start__continued.joblib
    save_snapshot_path=None,
    verbose=True,
)

print("continued snapshot:", resume_res["continued_snapshot_path"])

display(resume_res["daily_reward_df"].tail())
display(resume_res["daily_log_df"].tail())
display(resume_res["trades_df"].head())


[TRAIN][DAILY-PROB] n=3969 pos=890 (22.42%)
[TRAIN][DAILY-PROB] n=3994 pos=891 (22.31%)
[TRAIN][DAILY-PROB] n=4019 pos=898 (22.34%)
[TRAIN][DAILY-PROB] n=4044 pos=901 (22.28%)
[TRAIN][DAILY-PROB] n=4069 pos=909 (22.34%)
[TRAIN][DAILY-PROB] n=4094 pos=913 (22.30%)
[TRAIN][DAILY-PROB] n=4119 pos=917 (22.26%)
[TRAIN][DAILY-PROB] n=4144 pos=920 (22.20%)
[TRAIN][DAILY-PROB] n=4169 pos=928 (22.26%)
[TRAIN][DAILY-PROB] n=4194 pos=933 (22.25%)
[TRAIN][DAILY-PROB] n=4219 pos=937 (22.21%)
[TRAIN][DAILY-PROB] n=4244 pos=947 (22.31%)
[TRAIN][DAILY-PROB] n=4269 pos=955 (22.37%)
[TRAIN][DAILY-PROB] n=4294 pos=959 (22.33%)
[TRAIN][DAILY-PROB] n=4319 pos=963 (22.30%)
[TRAIN][DAILY-PROB] n=4344 pos=963 (22.17%)
[TRAIN][DAILY-PROB] n=4369 pos=970 (22.20%)
[TRAIN][DAILY-PROB] n=4394 pos=980 (22.30%)
[TRAIN][DAILY-PROB] n=4419 pos=988 (22.36%)
[TRAIN][DAILY-PROB] n=4444 pos=998 (22.46%)
[TRAIN][DAILY-PROB] n=4469 pos=1006 (22.51%)
[TRAIN][DAILY-PROB] n=4494 pos=1015 (22.59%)
[TRAIN][DAILY-PROB] n=4519 pos

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
4094,2026-04-15,0.125536,0.2,0.3,FORCE_BUY,0.008710,-0.000604,-0.000604,0.008710,FORCE_BUY,592023.898955,700.65,-0.50,2.00
4095,2026-04-16,0.119575,0.2,0.3,FORCE_BUY,0.001812,0.001528,-0.000257,0.001812,FORCE_BUY,593096.543725,702.22,-0.50,2.00
4096,2026-04-17,0.138861,0.2,0.3,FORCE_BUY,0.011427,0.001167,0.001167,0.011427,FORCE_BUY,599873.873595,710.75,0.85,2.45
4097,2026-04-20,0.138060,0.2,0.3,FORCE_BUY,0.004147,0.004147,0.004147,0.004147,FORCE_BUY,602361.461980,709.49,0.80,2.35
4098,2026-04-21,0.187429,0.2,0.3,FORCE_BUY,-0.003720,-0.003720,-0.003720,-0.003720,FORCE_BUY,600120.559185,707.00,0.80,2.35


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
572,2026-04-15,210746.488312,0.0,1,-0.50,2.00,0.125536,FORCE_BUY,0.2,0.3
573,2026-04-16,211218.724074,0.0,1,0.85,2.45,0.119575,FORCE_BUY,0.2,0.3
574,2026-04-17,213784.438118,0.0,1,0.80,2.35,0.138861,FORCE_BUY,0.2,0.3
575,2026-04-20,213405.446360,0.0,1,0.80,2.35,0.138060,FORCE_BUY,0.2,0.3
576,2026-04-21,212656.486457,0.0,1,0.80,2.35,0.187429,FORCE_BUY,0.2,0.3


,side,seen_idx,exec_px,qty,fee,reason,ts,pred,th,gate,pnl,entry_px,entry_idx
0,buy,643541,469.280,213.092397,0.0,5m BUY signal,2024-01-04 05:45:00,0.576991,0.45,FREE,NaN,NaN,NaN
1,sell,643961,469.080,213.092397,0.0,5m SELL signal,2024-01-08 09:30:00,7.518479,-0.50,FREE,-42.618479,469.280,643541.0
2,buy,643981,470.275,212.550915,0.0,5m BUY signal,2024-01-08 11:10:00,0.467693,0.35,FREE,NaN,NaN,NaN
3,sell,643991,471.080,212.550915,0.0,5m SELL signal,2024-01-08 12:00:00,8.529417,-0.50,FREE,171.103487,470.275,643981.0
4,buy,644317,473.810,211.326238,0.0,5m BUY signal,2024-01-10 08:00:00,1.297413,-0.50,FREE,NaN,NaN,NaN


In [10]:
from pathlib import Path
import pandas as pd

from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot,
)
from adaptive_trade_extensions import RollingThresholdConfig, make_threshold_grid

snapshot_path = Path("checkpoints") / "QQQ_adaptive_reward_fresh_start_15days.joblib"

common_kwargs = dict(
    daily_csv_path="DataAPI/data/QQQ_DAY.csv",
    k5m_csv_path="DataAPI/data/QQQ_5M.csv",
    code="QQQ",
    daily_chan_start="2008-01-01",
    accumulation_start="2010-01-01",
    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=5,
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={
        "vix_": "VIX.csv",
    },
    static_buy_level=0.20,
    static_sell_level=0.30,
    daily_threshold_config=RollingThresholdConfig(
        lookback_days=15,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    verbose=True,
)


In [11]:
snapshot_res = build_adaptive_reward_snapshot(
    snapshot_path=str(snapshot_path),
    snapshot_end_time="2023-12-31",
    output_dir="output_adaptive_reward_snapshot_build_QQQ_15days",
    **common_kwargs,
)

print(snapshot_res["snapshot_path"])
display(snapshot_res["daily_reward_df"].tail())
display(snapshot_res["daily_log_df"].tail())


[TRAIN][DAILY-PROB] n=200 pos=33 (16.50%)
[TRAIN][DAILY-PROB] n=225 pos=40 (17.78%)
[TRAIN][DAILY-PROB] n=250 pos=46 (18.40%)
[TRAIN][DAILY-PROB] n=275 pos=50 (18.18%)
[TRAIN][DAILY-PROB] n=300 pos=56 (18.67%)
[TRAIN][DAILY-PROB] n=325 pos=59 (18.15%)
[TRAIN][DAILY-PROB] n=350 pos=73 (20.86%)
[TRAIN][DAILY-PROB] n=375 pos=78 (20.80%)
[TRAIN][DAILY-PROB] n=400 pos=89 (22.25%)
[TRAIN][DAILY-PROB] n=425 pos=101 (23.76%)
[TRAIN][DAILY-PROB] n=450 pos=108 (24.00%)
[TRAIN][DAILY-PROB] n=475 pos=114 (24.00%)
[TRAIN][DAILY-PROB] n=500 pos=119 (23.80%)
[TRAIN][DAILY-PROB] n=525 pos=119 (22.67%)
[TRAIN][DAILY-PROB] n=550 pos=125 (22.73%)
[TRAIN][DAILY-PROB] n=575 pos=130 (22.61%)
[TRAIN][DAILY-PROB] n=600 pos=138 (23.00%)
[TRAIN][DAILY-PROB] n=625 pos=144 (23.04%)
[TRAIN][DAILY-PROB] n=650 pos=145 (22.31%)
[TRAIN][DAILY-PROB] n=675 pos=145 (21.48%)
[TRAIN][DAILY-PROB] n=700 pos=149 (21.29%)
[TRAIN][DAILY-PROB] n=725 pos=151 (20.83%)
[TRAIN][DAILY-PROB] n=750 pos=155 (20.67%)
[TRAIN][DAILY-PROB] 

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
3517,2023-12-22,0.164723,0.2,0.3,FORCE_BUY,0.003787,0.003836,0.003049,0.003787,FREE,2.003848e+11,408.20,1.8,1.9
3518,2023-12-26,0.160003,0.2,0.3,FORCE_BUY,0.003149,-0.000098,-0.000098,0.003149,FORCE_BUY,2.010158e+11,410.89,1.9,-0.5
3519,2023-12-27,0.139857,0.2,0.3,FORCE_BUY,0.002019,0.000827,0.000827,0.002019,FORCE_BUY,2.014217e+11,411.90,1.9,-0.5
3520,2023-12-28,0.197356,0.2,0.3,FORCE_BUY,-0.002109,0.001019,0.000412,-0.002109,FREE,2.016269e+11,411.68,1.9,-0.5
3521,2023-12-29,0.204446,0.2,0.3,FREE,-0.006967,-0.004135,0.000170,-0.004135,FORCE_SELL,2.016611e+11,409.06,1.9,-0.5


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
3517,2023-12-22,1.797983e+07,0.000000e+00,1,1.9,-0.5,0.164723,FORCE_BUY,0.2,0.3
3518,2023-12-26,1.809831e+07,0.000000e+00,1,1.9,-0.5,0.160003,FORCE_BUY,0.2,0.3
3519,2023-12-27,1.814280e+07,0.000000e+00,1,1.9,-0.5,0.139857,FORCE_BUY,0.2,0.3
3520,2023-12-28,1.813311e+07,0.000000e+00,1,1.9,-0.5,0.197356,FORCE_BUY,0.2,0.3
3521,2023-12-29,1.806910e+07,1.806910e+07,0,1.9,-0.5,0.204446,FREE,0.2,0.3


In [12]:
snapshot_path = Path("checkpoints") / "QQQ_adaptive_reward_fresh_start_15days.joblib"
resume_res = run_adaptive_reward_from_snapshot(
    snapshot_path=str(snapshot_path),
    end_time="2026-12-31",
    sim_start="2024-01-01",
    initial_capital=100000.0,
    fee_pct=0.0,
    output_dir="output_adaptive_reward_resumed_fresh_QQQ_15days",
    # Optional. If omitted, the code saves next to the source snapshot as
    # checkpoints/SPY_adaptive_reward_fresh_start__continued.joblib
    save_snapshot_path=None,
    verbose=True,
)

print("continued snapshot:", resume_res["continued_snapshot_path"])

display(resume_res["daily_reward_df"].tail())
display(resume_res["daily_log_df"].tail())
display(resume_res["trades_df"].head())


[TRAIN][DAILY-PROB] n=4002 pos=843 (21.06%)
[TRAIN][DAILY-PROB] n=4027 pos=844 (20.96%)
[TRAIN][DAILY-PROB] n=4052 pos=850 (20.98%)
[TRAIN][DAILY-PROB] n=4077 pos=854 (20.95%)
[TRAIN][DAILY-PROB] n=4102 pos=858 (20.92%)
[TRAIN][DAILY-PROB] n=4127 pos=862 (20.89%)
[TRAIN][DAILY-PROB] n=4152 pos=873 (21.03%)
[TRAIN][DAILY-PROB] n=4177 pos=876 (20.97%)
[TRAIN][DAILY-PROB] n=4202 pos=881 (20.97%)
[TRAIN][DAILY-PROB] n=4227 pos=885 (20.94%)
[TRAIN][DAILY-PROB] n=4252 pos=890 (20.93%)
[TRAIN][DAILY-PROB] n=4277 pos=900 (21.04%)
[TRAIN][DAILY-PROB] n=4302 pos=909 (21.13%)
[TRAIN][DAILY-PROB] n=4327 pos=913 (21.10%)
[TRAIN][DAILY-PROB] n=4352 pos=915 (21.02%)
[TRAIN][DAILY-PROB] n=4377 pos=915 (20.90%)
[TRAIN][DAILY-PROB] n=4402 pos=926 (21.04%)
[TRAIN][DAILY-PROB] n=4427 pos=930 (21.01%)
[TRAIN][DAILY-PROB] n=4452 pos=937 (21.05%)
[TRAIN][DAILY-PROB] n=4477 pos=944 (21.09%)
[TRAIN][DAILY-PROB] n=4502 pos=947 (21.04%)
[TRAIN][DAILY-PROB] n=4527 pos=953 (21.05%)
[TRAIN][DAILY-PROB] n=4552 pos=9

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
4094,2026-04-15,0.195553,0.2,0.3,FORCE_BUY,0.010239,0.001994,0.000000,0.010239,FORCE_BUY,1.422341e+06,638.3600,0.85,2.50
4095,2026-04-16,0.180557,0.2,0.3,FORCE_BUY,0.001439,0.004868,0.000094,0.001439,FREE,1.429265e+06,640.2400,0.85,2.50
4096,2026-04-17,0.189899,0.2,0.3,FORCE_BUY,0.013857,0.005735,0.002843,0.013857,FORCE_BUY,1.449070e+06,648.9700,0.85,2.50
4097,2026-04-20,0.150277,0.2,0.3,FORCE_BUY,0.005571,0.004654,0.002343,0.005571,FORCE_BUY,1.457143e+06,647.9701,0.85,2.50
4098,2026-04-21,0.181344,0.2,0.3,FORCE_BUY,-0.001865,0.002293,-0.000231,-0.001865,FREE,1.460485e+06,647.4900,-0.50,2.25


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
572,2026-04-15,295368.607460,0.0,1,0.85,2.50,0.195553,FORCE_BUY,0.2,0.3
573,2026-04-16,296238.481797,0.0,1,0.85,2.50,0.180557,FORCE_BUY,0.2,0.3
574,2026-04-17,300277.845077,0.0,1,0.85,2.50,0.189899,FORCE_BUY,0.2,0.3
575,2026-04-20,299815.192231,0.0,1,-0.50,2.25,0.150277,FORCE_BUY,0.2,0.3
576,2026-04-21,299593.050386,0.0,1,-0.50,2.25,0.181344,FORCE_BUY,0.2,0.3


,side,seen_idx,exec_px,qty,fee,reason,ts,pred,th,gate,pnl,entry_px,entry_idx
0,buy,581988,421.2401,237.394303,0.0,5m BUY signal,2024-01-22 13:25:00,1.132745,0.2,FREE,NaN,NaN,NaN
1,sell,581995,422.2700,237.394303,0.0,5m SELL signal,2024-01-22 14:00:00,1.855192,-0.5,FREE,244.492393,421.2401,581988.0
2,buy,581999,421.5700,237.788487,0.0,5m BUY signal,2024-01-22 14:20:00,1.110677,0.2,FREE,NaN,NaN,NaN
3,sell,582009,422.0600,237.788487,0.0,5m SELL signal,2024-01-22 15:10:00,1.354036,-0.5,FREE,116.516359,421.5700,581999.0
4,buy,582018,421.7200,237.980197,0.0,5m BUY signal,2024-01-22 15:55:00,0.827127,0.2,FREE,NaN,NaN,NaN


In [13]:
from pathlib import Path
import pandas as pd

from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot,
)
from adaptive_trade_extensions import RollingThresholdConfig, make_threshold_grid

snapshot_path = Path("checkpoints") / "SPY_adaptive_reward_fresh_start_90days.joblib"

common_kwargs = dict(
    daily_csv_path="DataAPI/data/SPY_DAY.csv",
    k5m_csv_path="DataAPI/data/SPY_5M.csv",
    code="SPY",
    daily_chan_start="2008-01-01",
    accumulation_start="2010-01-01",
    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=5,
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={
        "vix_": "VIX.csv",
    },
    static_buy_level=0.20,
    static_sell_level=0.30,
    daily_threshold_config=RollingThresholdConfig(
        lookback_days=90,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    verbose=True,
)


In [14]:
snapshot_res = build_adaptive_reward_snapshot(
    snapshot_path=str(snapshot_path),
    snapshot_end_time="2023-12-31",
    output_dir="output_adaptive_reward_snapshot_build_SPY_90days",
    **common_kwargs,
)

print(snapshot_res["snapshot_path"])
display(snapshot_res["daily_reward_df"].tail())
display(snapshot_res["daily_log_df"].tail())


[TRAIN][DAILY-PROB] n=200 pos=43 (21.50%)
[TRAIN][DAILY-PROB] n=225 pos=49 (21.78%)
[TRAIN][DAILY-PROB] n=250 pos=57 (22.80%)
[TRAIN][DAILY-PROB] n=275 pos=66 (24.00%)
[TRAIN][DAILY-PROB] n=300 pos=74 (24.67%)
[TRAIN][DAILY-PROB] n=325 pos=78 (24.00%)
[TRAIN][DAILY-PROB] n=350 pos=85 (24.29%)
[TRAIN][DAILY-PROB] n=375 pos=94 (25.07%)
[TRAIN][DAILY-PROB] n=400 pos=101 (25.25%)
[TRAIN][DAILY-PROB] n=425 pos=104 (24.47%)
[TRAIN][DAILY-PROB] n=450 pos=112 (24.89%)
[TRAIN][DAILY-PROB] n=475 pos=112 (23.58%)
[TRAIN][DAILY-PROB] n=500 pos=113 (22.60%)
[TRAIN][DAILY-PROB] n=525 pos=124 (23.62%)
[TRAIN][DAILY-PROB] n=550 pos=131 (23.82%)
[TRAIN][DAILY-PROB] n=575 pos=143 (24.87%)
[TRAIN][DAILY-PROB] n=600 pos=150 (25.00%)
[TRAIN][DAILY-PROB] n=625 pos=150 (24.00%)
[TRAIN][DAILY-PROB] n=650 pos=158 (24.31%)
[TRAIN][DAILY-PROB] n=675 pos=160 (23.70%)
[TRAIN][DAILY-PROB] n=700 pos=161 (23.00%)
[TRAIN][DAILY-PROB] n=725 pos=168 (23.17%)
[TRAIN][DAILY-PROB] n=750 pos=173 (23.07%)
[TRAIN][DAILY-PROB]

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
3517,2023-12-22,0.151024,0.165,0.190,FORCE_BUY,-0.003475,-0.002643,0.000000,-0.003475,FORCE_SELL,3.722095e+09,473.18,1.95,0.75
3518,2023-12-26,0.136806,0.165,0.225,FORCE_BUY,0.002487,0.002162,0.000190,0.002487,FORCE_BUY,3.731353e+09,475.59,1.55,-0.50
3519,2023-12-27,0.114499,0.165,0.225,FORCE_BUY,0.002649,0.004244,0.000105,0.002649,FREE,3.747190e+09,476.90,1.55,-0.50
3520,2023-12-28,0.139628,0.165,0.225,FORCE_BUY,0.000000,0.000189,-0.000210,0.000000,FREE,3.747897e+09,477.05,-0.50,1.40
3521,2023-12-29,0.154295,0.165,0.225,FORCE_BUY,-0.004881,-0.005516,0.000126,-0.004881,FORCE_SELL,3.748368e+09,475.07,1.80,1.40


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
3517,2023-12-22,1.459875e+06,0.0,1,1.55,-0.5,0.151024,FORCE_BUY,0.165,0.190
3518,2023-12-26,1.467310e+06,0.0,1,1.55,-0.5,0.136806,FORCE_BUY,0.165,0.225
3519,2023-12-27,1.471352e+06,0.0,1,-0.50,1.4,0.114499,FORCE_BUY,0.165,0.225
3520,2023-12-28,1.471814e+06,0.0,1,1.80,1.4,0.139628,FORCE_BUY,0.165,0.225
3521,2023-12-29,1.465706e+06,0.0,1,1.80,1.4,0.154295,FORCE_BUY,0.165,0.225


In [15]:
resume_res = run_adaptive_reward_from_snapshot(
    snapshot_path=str(snapshot_path),
    end_time="2026-12-31",
    sim_start="2024-01-01",
    initial_capital=100000.0,
    fee_pct=0.0,
    output_dir="output_adaptive_reward_resumed_fresh_SPY_90days",
    # Optional. If omitted, the code saves next to the source snapshot as
    # checkpoints/SPY_adaptive_reward_fresh_start__continued.joblib
    save_snapshot_path=None,
    verbose=True,
)

print("continued snapshot:", resume_res["continued_snapshot_path"])

display(resume_res["daily_reward_df"].tail())
display(resume_res["daily_log_df"].tail())
display(resume_res["trades_df"].head())


[TRAIN][DAILY-PROB] n=3969 pos=890 (22.42%)
[TRAIN][DAILY-PROB] n=3994 pos=891 (22.31%)
[TRAIN][DAILY-PROB] n=4019 pos=898 (22.34%)
[TRAIN][DAILY-PROB] n=4044 pos=901 (22.28%)
[TRAIN][DAILY-PROB] n=4069 pos=909 (22.34%)
[TRAIN][DAILY-PROB] n=4094 pos=913 (22.30%)
[TRAIN][DAILY-PROB] n=4119 pos=917 (22.26%)
[TRAIN][DAILY-PROB] n=4144 pos=920 (22.20%)
[TRAIN][DAILY-PROB] n=4169 pos=928 (22.26%)
[TRAIN][DAILY-PROB] n=4194 pos=933 (22.25%)
[TRAIN][DAILY-PROB] n=4219 pos=937 (22.21%)
[TRAIN][DAILY-PROB] n=4244 pos=947 (22.31%)
[TRAIN][DAILY-PROB] n=4269 pos=955 (22.37%)
[TRAIN][DAILY-PROB] n=4294 pos=959 (22.33%)
[TRAIN][DAILY-PROB] n=4319 pos=963 (22.30%)
[TRAIN][DAILY-PROB] n=4344 pos=963 (22.17%)
[TRAIN][DAILY-PROB] n=4369 pos=970 (22.20%)
[TRAIN][DAILY-PROB] n=4394 pos=980 (22.30%)
[TRAIN][DAILY-PROB] n=4419 pos=988 (22.36%)
[TRAIN][DAILY-PROB] n=4444 pos=998 (22.46%)
[TRAIN][DAILY-PROB] n=4469 pos=1006 (22.51%)
[TRAIN][DAILY-PROB] n=4494 pos=1015 (22.59%)
[TRAIN][DAILY-PROB] n=4519 pos

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
4094,2026-04-15,0.125592,0.15,0.245,FORCE_BUY,0.008710,-0.000604,-0.000604,0.008710,FORCE_BUY,590909.363220,700.65,-0.50,2.00
4095,2026-04-16,0.119636,0.15,0.245,FORCE_BUY,0.001812,0.001528,-0.000257,0.001812,FORCE_BUY,591979.988644,702.22,-0.50,2.00
4096,2026-04-17,0.138922,0.15,0.245,FORCE_BUY,0.011427,0.001167,0.001167,0.011427,FORCE_BUY,598744.559610,710.75,0.85,2.45
4097,2026-04-20,0.138204,0.15,0.245,FORCE_BUY,0.004147,0.004147,0.004147,0.004147,FORCE_BUY,601227.464897,709.49,0.80,2.35
4098,2026-04-21,0.187617,0.15,0.245,FREE,-0.003720,-0.003720,-0.003720,-0.003720,FORCE_BUY,598990.780793,707.00,0.80,2.35


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
572,2026-04-15,220822.165281,0.0,1,-0.50,2.00,0.125592,FORCE_BUY,0.15,0.245
573,2026-04-16,221316.978382,0.0,1,0.85,2.45,0.119636,FORCE_BUY,0.15,0.245
574,2026-04-17,224005.357844,0.0,1,0.80,2.35,0.138922,FORCE_BUY,0.15,0.245
575,2026-04-20,223608.246692,0.0,1,0.80,2.35,0.138204,FORCE_BUY,0.15,0.245
576,2026-04-21,222823.479417,0.0,1,0.80,2.35,0.187617,FREE,0.15,0.245


,side,seen_idx,exec_px,qty,fee,reason,ts,pred,th,gate,pnl,entry_px,entry_idx
0,buy,644657,476.30,209.951711,0.0,5m BUY signal,2024-01-12 04:50:00,0.097202,0.05,FREE,NaN,NaN,NaN
1,sell,644700,476.36,209.951711,0.0,5m SELL signal,2024-01-12 08:30:00,3.818075,-0.50,FREE,12.597103,476.3,644657.0
2,buy,644721,476.00,210.110498,0.0,5m BUY signal,2024-01-12 10:15:00,0.079194,0.05,FREE,NaN,NaN,NaN
3,sell,644733,476.51,210.110498,0.0,5m SELL signal,2024-01-12 11:15:00,3.214192,-0.50,FREE,107.156354,476.0,644721.0
4,buy,644796,476.02,210.326779,0.0,5m BUY signal,2024-01-12 16:30:00,0.179593,0.05,FREE,NaN,NaN,NaN


In [1]:
from pathlib import Path
import pandas as pd

from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot,
)
from adaptive_trade_extensions import RollingThresholdConfig, make_threshold_grid

snapshot_path = Path("checkpoints") / "QQQ_adaptive_reward_fresh_start_90days.joblib"

common_kwargs = dict(
    daily_csv_path="DataAPI/data/QQQ_DAY.csv",
    k5m_csv_path="DataAPI/data/QQQ_5M.csv",
    code="QQQ",
    daily_chan_start="2008-01-01",
    accumulation_start="2010-01-01",
    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=5,
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={
        "vix_": "VIX.csv",
    },
    static_buy_level=0.20,
    static_sell_level=0.30,
    daily_threshold_config=RollingThresholdConfig(
        lookback_days=90,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    verbose=True,
)


In [2]:
snapshot_res = build_adaptive_reward_snapshot(
    snapshot_path=str(snapshot_path),
    snapshot_end_time="2023-12-31",
    output_dir="output_adaptive_reward_snapshot_build_QQQ_90days",
    **common_kwargs,
)

print(snapshot_res["snapshot_path"])
display(snapshot_res["daily_reward_df"].tail())
display(snapshot_res["daily_log_df"].tail())


[TRAIN][DAILY-PROB] n=200 pos=33 (16.50%)
[TRAIN][DAILY-PROB] n=225 pos=40 (17.78%)
[TRAIN][DAILY-PROB] n=250 pos=46 (18.40%)
[TRAIN][DAILY-PROB] n=275 pos=50 (18.18%)
[TRAIN][DAILY-PROB] n=300 pos=56 (18.67%)
[TRAIN][DAILY-PROB] n=325 pos=59 (18.15%)
[TRAIN][DAILY-PROB] n=350 pos=73 (20.86%)
[TRAIN][DAILY-PROB] n=375 pos=78 (20.80%)
[TRAIN][DAILY-PROB] n=400 pos=89 (22.25%)
[TRAIN][DAILY-PROB] n=425 pos=101 (23.76%)
[TRAIN][DAILY-PROB] n=450 pos=108 (24.00%)
[TRAIN][DAILY-PROB] n=475 pos=114 (24.00%)
[TRAIN][DAILY-PROB] n=500 pos=119 (23.80%)
[TRAIN][DAILY-PROB] n=525 pos=119 (22.67%)
[TRAIN][DAILY-PROB] n=550 pos=125 (22.73%)
[TRAIN][DAILY-PROB] n=575 pos=130 (22.61%)
[TRAIN][DAILY-PROB] n=600 pos=138 (23.00%)
[TRAIN][DAILY-PROB] n=625 pos=144 (23.04%)
[TRAIN][DAILY-PROB] n=650 pos=145 (22.31%)
[TRAIN][DAILY-PROB] n=675 pos=145 (21.48%)
[TRAIN][DAILY-PROB] n=700 pos=149 (21.29%)
[TRAIN][DAILY-PROB] n=725 pos=151 (20.83%)
[TRAIN][DAILY-PROB] n=750 pos=155 (20.67%)
[TRAIN][DAILY-PROB] 

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
3517,2023-12-22,0.164809,0.22,0.28,FORCE_BUY,0.003787,0.003836,0.003049,0.003787,FREE,2.453586e+11,408.20,1.8,1.9
3518,2023-12-26,0.160100,0.22,0.28,FORCE_BUY,0.003149,-0.000098,-0.000098,0.003149,FORCE_BUY,2.461314e+11,410.89,1.9,-0.5
3519,2023-12-27,0.139934,0.22,0.30,FORCE_BUY,0.002019,0.000827,0.000827,0.002019,FORCE_BUY,2.466283e+11,411.90,1.9,-0.5
3520,2023-12-28,0.197314,0.22,0.30,FORCE_BUY,-0.002109,0.001019,0.000412,-0.002109,FREE,2.468795e+11,411.68,1.9,-0.5
3521,2023-12-29,0.204419,0.22,0.30,FORCE_BUY,-0.006967,-0.004135,0.000170,-0.006967,FORCE_SELL,2.469215e+11,409.06,1.9,-0.5


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
3517,2023-12-22,1.751273e+07,0.0,1,1.9,-0.5,0.164809,FORCE_BUY,0.22,0.28
3518,2023-12-26,1.762813e+07,0.0,1,1.9,-0.5,0.160100,FORCE_BUY,0.22,0.28
3519,2023-12-27,1.767146e+07,0.0,1,1.9,-0.5,0.139934,FORCE_BUY,0.22,0.30
3520,2023-12-28,1.766203e+07,0.0,1,1.9,-0.5,0.197314,FORCE_BUY,0.22,0.30
3521,2023-12-29,1.754962e+07,0.0,1,1.9,-0.5,0.204419,FORCE_BUY,0.22,0.30


In [3]:
resume_res = run_adaptive_reward_from_snapshot(
    snapshot_path=str(snapshot_path),
    end_time="2026-12-31",
    sim_start="2024-01-01",
    initial_capital=100000.0,
    fee_pct=0.0,
    output_dir="output_adaptive_reward_resumed_fresh_QQQ_90days",
    # Optional. If omitted, the code saves next to the source snapshot as
    # checkpoints/QQQ_adaptive_reward_fresh_start__continued.joblib
    save_snapshot_path=None,
    verbose=True,
)

print("continued snapshot:", resume_res["continued_snapshot_path"])

display(resume_res["daily_reward_df"].tail())
display(resume_res["daily_log_df"].tail())
display(resume_res["trades_df"].head())


[TRAIN][DAILY-PROB] n=4002 pos=843 (21.06%)
[TRAIN][DAILY-PROB] n=4027 pos=844 (20.96%)
[TRAIN][DAILY-PROB] n=4052 pos=850 (20.98%)
[TRAIN][DAILY-PROB] n=4077 pos=854 (20.95%)
[TRAIN][DAILY-PROB] n=4102 pos=858 (20.92%)
[TRAIN][DAILY-PROB] n=4127 pos=862 (20.89%)
[TRAIN][DAILY-PROB] n=4152 pos=873 (21.03%)
[TRAIN][DAILY-PROB] n=4177 pos=876 (20.97%)
[TRAIN][DAILY-PROB] n=4202 pos=881 (20.97%)
[TRAIN][DAILY-PROB] n=4227 pos=885 (20.94%)
[TRAIN][DAILY-PROB] n=4252 pos=890 (20.93%)
[TRAIN][DAILY-PROB] n=4277 pos=900 (21.04%)
[TRAIN][DAILY-PROB] n=4302 pos=909 (21.13%)
[TRAIN][DAILY-PROB] n=4327 pos=913 (21.10%)
[TRAIN][DAILY-PROB] n=4352 pos=915 (21.02%)
[TRAIN][DAILY-PROB] n=4377 pos=915 (20.90%)
[TRAIN][DAILY-PROB] n=4402 pos=926 (21.04%)
[TRAIN][DAILY-PROB] n=4427 pos=930 (21.01%)
[TRAIN][DAILY-PROB] n=4452 pos=937 (21.05%)
[TRAIN][DAILY-PROB] n=4477 pos=944 (21.09%)
[TRAIN][DAILY-PROB] n=4502 pos=947 (21.04%)
[TRAIN][DAILY-PROB] n=4527 pos=953 (21.05%)
[TRAIN][DAILY-PROB] n=4552 pos=9

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
4094,2026-04-15,0.195582,0.24,0.445,FORCE_BUY,0.010239,0.001994,0.000000,0.010239,FORCE_BUY,1.504969e+06,638.3600,0.85,2.50
4095,2026-04-16,0.180534,0.24,0.445,FORCE_BUY,0.001439,0.004868,0.000094,0.001439,FREE,1.512295e+06,640.2400,0.85,2.50
4096,2026-04-17,0.189910,0.24,0.445,FORCE_BUY,0.013857,0.005735,0.002843,0.013857,FORCE_BUY,1.533251e+06,648.9700,0.85,2.50
4097,2026-04-20,0.150180,0.24,0.445,FORCE_BUY,0.005571,0.004654,0.002343,0.005571,FORCE_BUY,1.541794e+06,647.9701,0.85,2.50
4098,2026-04-21,0.181258,0.34,0.445,FORCE_BUY,-0.001865,0.002293,-0.000231,-0.001865,FREE,1.545329e+06,647.4900,-0.50,2.25


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
572,2026-04-15,268998.496978,0.0,1,0.85,2.50,0.195582,FORCE_BUY,0.24,0.445
573,2026-04-16,269790.710109,0.0,1,0.85,2.50,0.180534,FORCE_BUY,0.24,0.445
574,2026-04-17,273469.444488,0.0,1,0.85,2.50,0.189910,FORCE_BUY,0.24,0.445
575,2026-04-20,273048.096664,0.0,1,-0.50,2.25,0.150180,FORCE_BUY,0.24,0.445
576,2026-04-21,272845.787343,0.0,1,-0.50,2.25,0.181258,FORCE_BUY,0.34,0.445


,side,seen_idx,exec_px,qty,fee,reason,ts,pred,th,gate,pnl,entry_px,entry_idx
0,buy,581988,421.2401,237.394303,0.0,5m BUY signal,2024-01-22 13:25:00,1.132745,0.2,FREE,NaN,NaN,NaN
1,sell,581995,422.2700,237.394303,0.0,5m SELL signal,2024-01-22 14:00:00,1.855192,-0.5,FREE,244.492393,421.2401,581988.0
2,buy,581999,421.5700,237.788487,0.0,5m BUY signal,2024-01-22 14:20:00,1.110677,0.2,FREE,NaN,NaN,NaN
3,sell,582009,422.0600,237.788487,0.0,5m SELL signal,2024-01-22 15:10:00,1.354036,-0.5,FREE,116.516359,421.5700,581999.0
4,buy,582018,421.7200,237.980197,0.0,5m BUY signal,2024-01-22 15:55:00,0.827127,0.2,FREE,NaN,NaN,NaN


In [1]:
from pathlib import Path
import pandas as pd

from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot,
)
from adaptive_trade_extensions import RollingThresholdConfig, make_threshold_grid

snapshot_path = Path("checkpoints") / "SPY_adaptive_reward_fresh_start_90days_at_2020.joblib"

common_kwargs = dict(
    daily_csv_path="DataAPI/data/SPY_DAY.csv",
    k5m_csv_path="DataAPI/data/SPY_5M.csv",
    code="SPY",
    daily_chan_start="2008-01-01",
    accumulation_start="2010-01-01",
    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=5,
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={
        "vix_": "VIX.csv",
    },
    static_buy_level=0.20,
    static_sell_level=0.30,
    daily_threshold_config=RollingThresholdConfig(
        lookback_days=252,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    verbose=True,
)


In [5]:
snapshot_res = build_adaptive_reward_snapshot(
    snapshot_path=str(snapshot_path),
    snapshot_end_time="2019-12-31",
    output_dir="output_adaptive_reward_snapshot_build_SPY_90days_at_2020",
    **common_kwargs,
)

print(snapshot_res["snapshot_path"])
display(snapshot_res["daily_reward_df"].tail())
display(snapshot_res["daily_log_df"].tail())


[TRAIN][DAILY-PROB] n=200 pos=43 (21.50%)
[TRAIN][DAILY-PROB] n=225 pos=49 (21.78%)
[TRAIN][DAILY-PROB] n=250 pos=57 (22.80%)
[TRAIN][DAILY-PROB] n=275 pos=66 (24.00%)
[TRAIN][DAILY-PROB] n=300 pos=74 (24.67%)
[TRAIN][DAILY-PROB] n=325 pos=78 (24.00%)
[TRAIN][DAILY-PROB] n=350 pos=85 (24.29%)
[TRAIN][DAILY-PROB] n=375 pos=94 (25.07%)
[TRAIN][DAILY-PROB] n=400 pos=101 (25.25%)
[TRAIN][DAILY-PROB] n=425 pos=104 (24.47%)
[TRAIN][DAILY-PROB] n=450 pos=112 (24.89%)
[TRAIN][DAILY-PROB] n=475 pos=112 (23.58%)
[TRAIN][DAILY-PROB] n=500 pos=113 (22.60%)
[TRAIN][DAILY-PROB] n=525 pos=124 (23.62%)
[TRAIN][DAILY-PROB] n=550 pos=131 (23.82%)
[TRAIN][DAILY-PROB] n=575 pos=143 (24.87%)
[TRAIN][DAILY-PROB] n=600 pos=150 (25.00%)
[TRAIN][DAILY-PROB] n=625 pos=150 (24.00%)
[TRAIN][DAILY-PROB] n=650 pos=158 (24.31%)
[TRAIN][DAILY-PROB] n=675 pos=160 (23.70%)
[TRAIN][DAILY-PROB] n=700 pos=161 (23.00%)
[TRAIN][DAILY-PROB] n=725 pos=168 (23.17%)
[TRAIN][DAILY-PROB] n=750 pos=173 (23.07%)
[TRAIN][DAILY-PROB]

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
2510,2019-12-23,0.147412,0.155,0.18,FORCE_BUY,0.000031,0.000031,0.000031,0.000031,FORCE_BUY,5.100441e+07,321.35,1.25,1.95
2511,2019-12-24,0.136711,0.155,0.18,FORCE_BUY,-0.000715,-0.000715,-0.000715,-0.000715,FORCE_BUY,5.096792e+07,321.24,1.25,1.95
2512,2019-12-26,0.125245,0.155,0.18,FORCE_BUY,0.005754,0.000653,0.000653,0.005754,FORCE_BUY,5.126117e+07,323.39,1.50,-0.50
2513,2019-12-27,0.156382,0.155,0.18,FREE,-0.003770,-0.001997,-0.000031,-0.001997,FORCE_SELL,5.125958e+07,322.39,1.50,-0.50
2514,2019-12-30,0.191498,0.155,0.18,FORCE_SELL,-0.005354,0.002557,-0.000093,-0.000093,FREE,5.139065e+07,321.38,1.50,-0.50


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
2510,2019-12-23,389266.697318,0.000000,1,1.25,1.95,0.147412,FORCE_BUY,0.155,0.18
2511,2019-12-24,389133.449032,0.000000,1,1.50,-0.50,0.136711,FORCE_BUY,0.155,0.18
2512,2019-12-26,391737.847349,0.000000,1,1.50,-0.50,0.125245,FORCE_BUY,0.155,0.18
2513,2019-12-27,391221.655202,0.000000,1,1.50,-0.50,0.156382,FREE,0.155,0.18
2514,2019-12-30,392058.973178,392058.973178,0,1.50,-0.50,0.191498,FORCE_SELL,0.155,0.18


In [2]:
resume_res = run_adaptive_reward_from_snapshot(
    dp_lookback_override=252,
    snapshot_path=str(snapshot_path),
    end_time="2026-12-31",
    sim_start="2020-01-01",
    initial_capital=100000.0,
    fee_pct=0.0,
    output_dir="output_adaptive_reward_resumed_fresh_SPY_252days_at_2020",
    # Optional. If omitted, the code saves next to the source snapshot as
    # checkpoints/SPY_adaptive_reward_fresh_start__continued.joblib
    save_snapshot_path=None,
    verbose=True,
)

print("continued snapshot:", resume_res["continued_snapshot_path"])

display(resume_res["daily_reward_df"].tail())
display(resume_res["daily_log_df"].tail())
display(resume_res["trades_df"].head())


[TRAIN][DAILY-PROB] n=2963 pos=658 (22.21%)
[TRAIN][DAILY-PROB] n=2988 pos=668 (22.36%)
[TRAIN][DAILY-PROB] n=3013 pos=683 (22.67%)
[TRAIN][DAILY-PROB] n=3038 pos=690 (22.71%)
[TRAIN][DAILY-PROB] n=3063 pos=696 (22.72%)
[TRAIN][DAILY-PROB] n=3088 pos=697 (22.57%)
[TRAIN][DAILY-PROB] n=3113 pos=698 (22.42%)
[TRAIN][DAILY-PROB] n=3138 pos=704 (22.43%)
[TRAIN][DAILY-PROB] n=3163 pos=714 (22.57%)
[TRAIN][DAILY-PROB] n=3188 pos=715 (22.43%)
[TRAIN][DAILY-PROB] n=3213 pos=718 (22.35%)
[TRAIN][DAILY-PROB] n=3238 pos=725 (22.39%)
[TRAIN][DAILY-PROB] n=3263 pos=727 (22.28%)
[TRAIN][DAILY-PROB] n=3288 pos=730 (22.20%)
[TRAIN][DAILY-PROB] n=3313 pos=732 (22.09%)
[TRAIN][DAILY-PROB] n=3338 pos=733 (21.96%)
[TRAIN][DAILY-PROB] n=3363 pos=733 (21.80%)
[TRAIN][DAILY-PROB] n=3388 pos=747 (22.05%)
[TRAIN][DAILY-PROB] n=3413 pos=749 (21.95%)
[TRAIN][DAILY-PROB] n=3438 pos=753 (21.90%)
[TRAIN][DAILY-PROB] n=3463 pos=762 (22.00%)
[TRAIN][DAILY-PROB] n=3488 pos=770 (22.08%)
[TRAIN][DAILY-PROB] n=3513 pos=7

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
4094,2026-04-15,0.122837,0.155,0.315,FORCE_BUY,0.008710,-0.000604,-0.000604,0.008710,FORCE_BUY,3.626008e+07,700.65,-0.50,1.7
4095,2026-04-16,0.117469,0.155,0.315,FORCE_BUY,0.001812,0.001528,-0.000257,0.001812,FORCE_BUY,3.632578e+07,702.22,-0.50,1.7
4096,2026-04-17,0.136876,0.155,0.315,FORCE_BUY,0.011427,0.000668,-0.000014,0.011427,FORCE_BUY,3.674087e+07,710.75,1.15,-0.5
4097,2026-04-20,0.134608,0.155,0.315,FORCE_BUY,0.004147,0.004147,0.004147,0.004147,FORCE_BUY,3.689323e+07,709.49,1.15,-0.5
4098,2026-04-21,0.184940,0.155,0.315,FREE,-0.003720,0.001227,0.001860,0.001227,FORCE_SELL,3.696186e+07,707.00,1.15,-0.5


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
1579,2026-04-15,624999.137145,0.000000,1,-0.50,1.7,0.122837,FORCE_BUY,0.155,0.315
1580,2026-04-16,626399.620476,0.000000,1,1.15,-0.5,0.117469,FORCE_BUY,0.155,0.315
1581,2026-04-17,634008.615894,0.000000,1,1.15,-0.5,0.136876,FORCE_BUY,0.155,0.315
1582,2026-04-20,632884.661119,0.000000,1,1.15,-0.5,0.134608,FORCE_BUY,0.155,0.315
1583,2026-04-21,634469.654019,634469.654019,0,1.15,-0.5,0.184940,FREE,0.155,0.315


,side,seen_idx,exec_px,qty,fee,reason,ts,pred,th,gate,pnl,entry_px,entry_idx
0,buy,452961,320.23,312.275552,0.0,5m BUY signal,2020-01-06 04:00:00,0.306116,-0.5,FREE,NaN,NaN,NaN
1,sell,453104,323.66,312.275552,0.0,5m SELL signal,2020-01-06 15:55:00,4.976673,-0.5,FREE,1071.105143,320.23,452961.0
2,buy,453119,323.26,312.661960,0.0,5m BUY signal,2020-01-06 17:10:00,-0.414623,-0.5,FREE,NaN,NaN,NaN
3,sell,453130,323.44,312.661960,0.0,5m SELL signal,2020-01-06 18:10:00,4.249341,-0.5,FREE,56.279153,323.26,453119.0
4,buy,453136,323.23,312.865094,0.0,5m BUY signal,2020-01-06 18:45:00,-0.434716,-0.5,FREE,NaN,NaN,NaN


In [1]:

from pathlib import Path
import pandas as pd

from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot,
)
from adaptive_trade_extensions import RollingThresholdConfig, make_threshold_grid

snapshot_path = Path("checkpoints") / "TQQQ_adaptive_reward_fresh_start_252days_at_2020.joblib"

common_kwargs = dict(
    daily_csv_path="DataAPI/data/TQQQ_day.csv",
    k5m_csv_path="DataAPI/data/TQQQ_5M.csv",
    code="TQQQ",
    daily_chan_start="2010-02-11",
    accumulation_start="2012-02-11",
    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=5,
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={
        "vix_": "VIX.csv",
    },
    static_buy_level=0.20,
    static_sell_level=0.30,
    daily_threshold_config=RollingThresholdConfig(
        lookback_days=252,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    verbose=True,
)


In [6]:
snapshot_res = build_adaptive_reward_snapshot(
    snapshot_path=str(snapshot_path),
    snapshot_end_time="2019-12-31",
    output_dir="output_adaptive_reward_snapshot_build_TQQQ_252days_at_2020",
    **common_kwargs,
)

print(snapshot_res["snapshot_path"])
display(snapshot_res["daily_reward_df"].tail())
display(snapshot_res["daily_log_df"].tail())

[TRAIN][DAILY-PROB] n=200 pos=49 (24.50%)
[TRAIN][DAILY-PROB] n=225 pos=52 (23.11%)
[TRAIN][DAILY-PROB] n=250 pos=61 (24.40%)
[TRAIN][DAILY-PROB] n=275 pos=65 (23.64%)
[TRAIN][DAILY-PROB] n=300 pos=76 (25.33%)
[TRAIN][DAILY-PROB] n=325 pos=76 (23.38%)
[TRAIN][DAILY-PROB] n=350 pos=76 (21.71%)
[TRAIN][DAILY-PROB] n=375 pos=79 (21.07%)
[TRAIN][DAILY-PROB] n=400 pos=84 (21.00%)
[TRAIN][DAILY-PROB] n=425 pos=91 (21.41%)
[TRAIN][DAILY-PROB] n=450 pos=96 (21.33%)
[TRAIN][DAILY-PROB] n=475 pos=99 (20.84%)
[TRAIN][DAILY-PROB] n=500 pos=104 (20.80%)
[TRAIN][DAILY-PROB] n=525 pos=114 (21.71%)
[TRAIN][DAILY-PROB] n=550 pos=118 (21.45%)
[TRAIN][DAILY-PROB] n=575 pos=120 (20.87%)
[TRAIN][DAILY-PROB] n=600 pos=128 (21.33%)
[TRAIN][DAILY-PROB] n=625 pos=137 (21.92%)
[TRAIN][DAILY-PROB] n=650 pos=142 (21.85%)
[TRAIN][DAILY-PROB] n=675 pos=150 (22.22%)
[TRAIN][DAILY-PROB] n=700 pos=152 (21.71%)
[TRAIN][DAILY-PROB] n=725 pos=155 (21.38%)
[TRAIN][DAILY-PROB] n=750 pos=160 (21.33%)
[TRAIN][DAILY-PROB] n=7

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
1978,2019-12-23,0.149304,0.205,0.230,FORCE_BUY,0.002572,0.004443,0.003161,0.002572,FREE,1.338595e+13,10.7212,-0.5,-0.50
1979,2019-12-24,0.140908,0.205,0.230,FORCE_BUY,0.000121,-0.000571,0.002106,0.000121,FORCE_SELL,1.341415e+13,10.7325,-0.5,-0.50
1980,2019-12-26,0.153845,0.205,0.230,FORCE_BUY,0.029743,0.006162,0.001980,0.029743,FORCE_BUY,1.381313e+13,11.0787,-0.5,1.95
1981,2019-12-27,0.201983,0.205,0.230,FORCE_BUY,-0.014070,0.000797,-0.000117,-0.014070,FREE,1.382413e+13,10.9525,-0.5,1.95
1982,2019-12-30,0.226911,0.190,0.215,FORCE_SELL,-0.021668,-0.029709,-0.027342,-0.027342,FORCE_BUY,1.352459e+13,10.7775,-0.5,1.95


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
1978,2019-12-23,3.109785e+07,0.000000e+00,1,-0.5,-0.50,0.149304,FORCE_BUY,0.205,0.230
1979,2019-12-24,3.113063e+07,0.000000e+00,1,-0.5,1.95,0.140908,FORCE_BUY,0.205,0.230
1980,2019-12-26,3.213481e+07,0.000000e+00,1,-0.5,1.95,0.153845,FORCE_BUY,0.205,0.230
1981,2019-12-27,3.176876e+07,0.000000e+00,1,-0.5,1.95,0.201983,FORCE_BUY,0.205,0.230
1982,2019-12-30,3.107987e+07,3.107987e+07,0,-0.5,1.95,0.226911,FORCE_SELL,0.190,0.215


In [2]:
resume_res = run_adaptive_reward_from_snapshot(
    dp_lookback_override=252,
    snapshot_path=str(snapshot_path),
    end_time="2026-12-31",
    sim_start="2020-01-01",
    initial_capital=100000.0,
    fee_pct=0.0,
    output_dir="output_adaptive_reward_resumed_fresh_TQQQ_252days_at_2020",
    # Optional. If omitted, the code saves next to the source snapshot as
    # checkpoints/TQQQ_adaptive_reward_fresh_start__continued.joblib
    save_snapshot_path="output_adaptive_reward_resumed_fresh_TQQQ_252days_at_2020/final_checkpoint.joblib",
    autosave_year_start_checkpoints=True,
    verbose=True,
)

print("continued snapshot:", resume_res["continued_snapshot_path"])

display(resume_res["daily_reward_df"].tail())
display(resume_res["daily_log_df"].tail())
display(resume_res["trades_df"].head())


[TRAIN][DAILY-PROB] n=2332 pos=470 (20.15%)
[TRAIN][DAILY-PROB] n=2357 pos=475 (20.15%)
[TRAIN][DAILY-PROB] n=2382 pos=477 (20.03%)
[TRAIN][DAILY-PROB] n=2407 pos=478 (19.86%)
[TRAIN][DAILY-PROB] n=2432 pos=480 (19.74%)
[TRAIN][DAILY-PROB] n=2457 pos=484 (19.70%)
[TRAIN][DAILY-PROB] n=2482 pos=487 (19.62%)
[TRAIN][DAILY-PROB] n=2507 pos=493 (19.66%)
[TRAIN][DAILY-PROB] n=2532 pos=504 (19.91%)
[TRAIN][DAILY-PROB] n=2557 pos=505 (19.75%)
[TRAIN][DAILY-PROB] n=2582 pos=508 (19.67%)
[TRAIN][DAILY-PROB] n=2607 pos=516 (19.79%)
[TRAIN][DAILY-PROB] n=2632 pos=519 (19.72%)
[TRAIN][DAILY-PROB] n=2657 pos=525 (19.76%)
[TRAIN][DAILY-PROB] n=2682 pos=525 (19.57%)
[TRAIN][DAILY-PROB] n=2707 pos=528 (19.50%)
[TRAIN][DAILY-PROB] n=2732 pos=531 (19.44%)
[TRAIN][DAILY-PROB] n=2757 pos=542 (19.66%)
[TRAIN][DAILY-PROB] n=2782 pos=545 (19.59%)
[TRAIN][DAILY-PROB] n=2807 pos=551 (19.63%)
[TRAIN][DAILY-PROB] n=2832 pos=557 (19.67%)
[TRAIN][DAILY-PROB] n=2857 pos=559 (19.57%)
[TRAIN][DAILY-PROB] n=2882 pos=5

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
3585,2026-05-18,0.259901,0.24,0.26,FREE,0.002692,0.046294,0.013190,0.046294,FREE,4.418697e+16,74.50,-0.5,-0.5
3586,2026-05-19,0.222622,0.24,0.26,FORCE_BUY,-0.015875,-0.023606,-0.014933,-0.015875,FORCE_SELL,4.352711e+16,73.15,-0.5,-0.5
3587,2026-05-20,0.201810,0.24,0.26,FORCE_BUY,0.017227,0.006506,0.004069,0.017227,FORCE_BUY,4.427696e+16,74.99,-0.5,-0.5
3588,2026-05-21,0.216030,0.24,0.26,FORCE_BUY,0.022632,0.029939,0.002500,0.022632,FREE,4.560255e+16,77.72,-0.5,-0.5
3589,2026-05-22,0.241934,0.24,0.26,FREE,-0.009860,0.002904,0.000768,0.002904,FREE,4.573496e+16,77.32,-0.5,-0.5


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
1602,2026-05-18,1.163219e+09,0.000000e+00,1,-0.5,-0.5,0.259901,FREE,0.24,0.26
1603,2026-05-19,1.142140e+09,0.000000e+00,1,-0.5,-0.5,0.222622,FORCE_BUY,0.24,0.26
1604,2026-05-20,1.170869e+09,0.000000e+00,1,-0.5,-0.5,0.201810,FORCE_BUY,0.24,0.26
1605,2026-05-21,1.213495e+09,0.000000e+00,1,-0.5,-0.5,0.216030,FORCE_BUY,0.24,0.26
1606,2026-05-22,1.222812e+09,1.222812e+09,0,-0.5,-0.5,0.241934,FREE,0.24,0.26


,side,seen_idx,exec_px,qty,fee,reason,ts,pred,th,gate,pnl,entry_px,entry_idx
0,buy,253880,11.2613,8879.969453,0.0,ADAPTIVE_FORCE_BUY->first acceptable 5m signal,2020-01-07 08:30:00,2.143581,-0.5,FORCE_BUY,NaN,NaN,NaN
1,sell,254029,11.2100,8879.969453,0.0,5m SELL signal,2020-01-08 05:15:00,26.507231,-0.5,FREE,-455.542433,11.2613,253880.0
2,buy,254050,11.2625,8838.575589,0.0,5m BUY signal,2020-01-08 07:00:00,0.846646,-0.5,FREE,NaN,NaN,NaN
3,sell,254065,11.3375,8838.575589,0.0,5m SELL signal,2020-01-08 08:15:00,23.341526,-0.5,FREE,662.893169,11.2625,254050.0
4,buy,254070,11.2875,8877.727640,0.0,5m BUY signal,2020-01-08 08:40:00,2.161747,-0.5,FREE,NaN,NaN,NaN


In [1]:

from pathlib import Path
import pandas as pd

from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot,
)
from adaptive_trade_extensions import RollingThresholdConfig, make_threshold_grid

snapshot_path = Path("checkpoints") / "SOXL_adaptive_reward_fresh_start_252days_at_2020.joblib"

common_kwargs = dict(
    daily_csv_path="DataAPI/data/SOXL_day.csv",
    k5m_csv_path="DataAPI/data/SOXL_5M.csv",
    code="SOXL",
    daily_chan_start="2010-02-11",
    accumulation_start="2012-02-11",
    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=5,
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={
        "vix_": "VIX.csv",
    },
    static_buy_level=0.20,
    static_sell_level=0.30,
    daily_threshold_config=RollingThresholdConfig(
        lookback_days=252,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    verbose=True,
)


In [2]:
snapshot_res = build_adaptive_reward_snapshot(
    snapshot_path=str(snapshot_path),
    snapshot_end_time="2019-12-31",
    output_dir="output_adaptive_reward_snapshot_build_TQQQ_252days_at_2020",
    **common_kwargs,
)

print("base snapshot:", snapshot_res["snapshot_path"])
print("year-start build checkpoints:")
for p in snapshot_res["year_start_checkpoint_paths"]:
    print(" ", p)

display(snapshot_res["daily_reward_df"].tail())
display(snapshot_res["daily_log_df"].tail())

[TRAIN][DAILY-PROB] n=200 pos=43 (21.50%)
[TRAIN][DAILY-PROB] n=225 pos=51 (22.67%)
[TRAIN][DAILY-PROB] n=250 pos=54 (21.60%)
[TRAIN][DAILY-PROB] n=275 pos=61 (22.18%)
[TRAIN][DAILY-PROB] n=300 pos=66 (22.00%)
[TRAIN][DAILY-PROB] n=325 pos=72 (22.15%)
[TRAIN][DAILY-PROB] n=350 pos=80 (22.86%)
[TRAIN][DAILY-PROB] n=375 pos=92 (24.53%)
[TRAIN][DAILY-PROB] n=400 pos=96 (24.00%)
[TRAIN][DAILY-PROB] n=425 pos=107 (25.18%)
[TRAIN][DAILY-PROB] n=450 pos=112 (24.89%)
[TRAIN][DAILY-PROB] n=475 pos=119 (25.05%)
[TRAIN][DAILY-PROB] n=500 pos=131 (26.20%)
[TRAIN][DAILY-PROB] n=525 pos=133 (25.33%)
[TRAIN][DAILY-PROB] n=550 pos=135 (24.55%)
[TRAIN][DAILY-PROB] n=575 pos=136 (23.65%)
[TRAIN][DAILY-PROB] n=600 pos=145 (24.17%)
[TRAIN][DAILY-PROB] n=625 pos=149 (23.84%)
[TRAIN][DAILY-PROB] n=650 pos=153 (23.54%)
[TRAIN][DAILY-PROB] n=675 pos=158 (23.41%)
[TRAIN][DAILY-PROB] n=700 pos=161 (23.00%)
[TRAIN][DAILY-PROB] n=725 pos=168 (23.17%)
[TRAIN][DAILY-PROB] n=750 pos=175 (23.33%)
[TRAIN][DAILY-PROB] 

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
1978,2019-12-23,0.231515,0.255,0.275,FORCE_BUY,-0.002679,-0.002679,-0.002679,-0.002679,FORCE_BUY,1.094626e+13,18.6167,-0.5,-0.5
1979,2019-12-24,0.194711,0.255,0.275,FORCE_BUY,0.003075,0.003075,0.003075,0.003075,FORCE_BUY,1.097992e+13,18.7241,-0.5,-0.5
1980,2019-12-26,0.202051,0.255,0.275,FORCE_BUY,0.003874,0.003874,0.003874,0.003874,FORCE_BUY,1.102246e+13,18.8400,-0.5,-0.5
1981,2019-12-27,0.222002,0.255,0.275,FORCE_BUY,-0.022254,-0.002908,0.006008,-0.022254,FORCE_SELL,1.108869e+13,18.4533,-0.5,-0.5
1982,2019-12-30,0.231132,0.255,0.275,FORCE_BUY,-0.015162,0.026836,0.005415,-0.015162,FREE,1.138627e+13,18.1867,-0.5,-0.5


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
1978,2019-12-23,4.715208e+06,0.0,1,-0.5,-0.5,0.231515,FORCE_BUY,0.255,0.275
1979,2019-12-24,4.742410e+06,0.0,1,-0.5,-0.5,0.194711,FORCE_BUY,0.255,0.275
1980,2019-12-26,4.771765e+06,0.0,1,-0.5,-0.5,0.202051,FORCE_BUY,0.255,0.275
1981,2019-12-27,4.673823e+06,0.0,1,-0.5,-0.5,0.222002,FORCE_BUY,0.255,0.275
1982,2019-12-30,4.606299e+06,0.0,1,-0.5,-0.5,0.231132,FORCE_BUY,0.255,0.275


In [ ]:
resume_res = run_adaptive_reward_from_snapshot(
    snapshot_path=str(snapshot_path),
    end_time="2026-12-31",
    sim_start="2020-01-01",
    initial_capital=100000.0,
    fee_pct=0.0,
    output_dir="output_adaptive_reward_resumed_fresh_TQQQ_252days_at_2020",
    save_snapshot_path=None,
    autosave_year_start_checkpoints=True,
    verbose=True,
)

print("continued snapshot:", resume_res["continued_snapshot_path"])
print("year-start resume checkpoints:")
for p in resume_res["year_start_checkpoint_paths"]:
    print(" ", p)

display(resume_res["daily_reward_df"].tail())
display(resume_res["daily_log_df"].tail())
display(resume_res["trades_df"].head())

[TRAIN][DAILY-PROB] n=2310 pos=525 (22.73%)
[TRAIN][DAILY-PROB] n=2335 pos=535 (22.91%)
[TRAIN][DAILY-PROB] n=2360 pos=542 (22.97%)
[TRAIN][DAILY-PROB] n=2385 pos=545 (22.85%)
[TRAIN][DAILY-PROB] n=2410 pos=548 (22.74%)
[TRAIN][DAILY-PROB] n=2435 pos=550 (22.59%)
[TRAIN][DAILY-PROB] n=2460 pos=551 (22.40%)
[TRAIN][DAILY-PROB] n=2485 pos=555 (22.33%)
[TRAIN][DAILY-PROB] n=2510 pos=565 (22.51%)
[TRAIN][DAILY-PROB] n=2535 pos=569 (22.45%)
[TRAIN][DAILY-PROB] n=2560 pos=574 (22.42%)
[TRAIN][DAILY-PROB] n=2585 pos=582 (22.51%)
[TRAIN][DAILY-PROB] n=2610 pos=587 (22.49%)
[TRAIN][DAILY-PROB] n=2635 pos=597 (22.66%)
[TRAIN][DAILY-PROB] n=2660 pos=602 (22.63%)
[TRAIN][DAILY-PROB] n=2685 pos=610 (22.72%)
[TRAIN][DAILY-PROB] n=2710 pos=620 (22.88%)
[TRAIN][DAILY-PROB] n=2735 pos=629 (23.00%)
[TRAIN][DAILY-PROB] n=2760 pos=630 (22.83%)
[TRAIN][DAILY-PROB] n=2785 pos=635 (22.80%)
[TRAIN][DAILY-PROB] n=2810 pos=643 (22.88%)
[TRAIN][DAILY-PROB] n=2835 pos=648 (22.86%)
[TRAIN][DAILY-PROB] n=2860 pos=6

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
3578,2026-05-07,0.434549,0.305,0.33,FORCE_SELL,-0.051580,0.001741,0.0,0.0,FREE,1.141265e+20,152.4300,-0.5,-0.5
3579,2026-05-08,0.407557,0.305,0.33,FORCE_SELL,0.113694,-0.004800,0.0,0.0,FORCE_BUY,1.271020e+20,178.6700,-0.5,-0.5
3580,2026-05-11,0.430346,0.305,0.33,FORCE_SELL,0.097270,0.082189,0.0,0.0,FORCE_BUY,1.394652e+20,190.5300,-0.5,-0.5
3581,2026-05-12,0.554729,0.305,0.33,FORCE_SELL,-0.068091,-0.005622,0.0,0.0,FORCE_SELL,1.394652e+20,167.6505,-0.5,-0.5
3582,2026-05-13,0.502656,0.305,0.33,FORCE_SELL,0.028616,0.089799,0.0,0.0,FREE,1.519891e+20,188.0000,-0.5,-0.5


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
1595,2026-05-07,3.629903e+08,3.629903e+08,0,-0.5,-0.5,0.434549,FORCE_SELL,0.305,0.33
1596,2026-05-08,3.629903e+08,3.629903e+08,0,-0.5,-0.5,0.407557,FORCE_SELL,0.305,0.33
1597,2026-05-11,3.629903e+08,3.629903e+08,0,-0.5,-0.5,0.430346,FORCE_SELL,0.305,0.33
1598,2026-05-12,3.629903e+08,3.629903e+08,0,-0.5,-0.5,0.554729,FORCE_SELL,0.305,0.33
1599,2026-05-13,3.629903e+08,3.629903e+08,0,-0.5,-0.5,0.502656,FORCE_SELL,0.305,0.33


,side,seen_idx,exec_px,qty,fee,reason,ts,pred,th,gate,pnl,entry_px,entry_idx
0,buy,163608,18.7813,5324.445060,0.0,5m BUY signal,2020-01-07 15:50:00,2.407178,-0.50,FREE,NaN,NaN,NaN
1,sell,163638,18.7847,5324.445060,0.0,5m SELL signal,2020-01-08 06:25:00,27.777378,-0.50,FREE,18.103113,18.7813,163608.0
2,buy,163733,18.7980,5320.677897,0.0,5m BUY signal,2020-01-08 15:45:00,2.382491,1.85,FREE,NaN,NaN,NaN
3,sell,163744,18.9267,5320.677897,0.0,5m SELL signal,2020-01-08 19:10:00,20.199308,-0.50,FREE,684.771245,18.7980,163733.0
4,buy,163772,19.4660,5173.270028,0.0,ADAPTIVE_FORCE_BUY->first acceptable 5m signal,2020-01-09 09:25:00,2.685045,-0.05,FORCE_BUY,NaN,NaN,NaN


In [1]:
from adaptive_reward_checkpoint_fresh import run_adaptive_reward_realtime_from_checkpoint

realtime_res = run_adaptive_reward_realtime_from_checkpoint(
    checkpoint_path="output_adaptive_reward_resumed_fresh_TQQQ_252days_at_2020/final_checkpoint.joblib",
    output_dir="output_adaptive_reward_realtime_TQQQ",
    end_time=None,  # None = now
    initial_capital=100000.0,
    fee_pct=0.0,
    save_snapshot_path="output_adaptive_reward_realtime_TQQQ/final_checkpoint.joblib",
    refresh_data=True,
    verbose=True,
)

print("new checkpoint:", realtime_res["continued_snapshot_path"])
display(realtime_res["trades_df"].tail())
display(realtime_res["daily_log_df"].tail())

[SAVED] output_adaptive_reward_realtime_TQQQ\trades.csv
[SAVED] output_adaptive_reward_realtime_TQQQ\daily_log.csv
[SAVED] output_adaptive_reward_realtime_TQQQ\daily_reward_log.csv
[SAVED] output_adaptive_reward_realtime_TQQQ\equity_vs_buyhold.png
[SAVED] output_adaptive_reward_realtime_TQQQ\price_with_trades.png
[SAVED] output_adaptive_reward_realtime_TQQQ\p_day.png
[CHECKPOINT] saved continued snapshot -> output_adaptive_reward_realtime_TQQQ/final_checkpoint.joblib
new checkpoint: output_adaptive_reward_realtime_TQQQ/final_checkpoint.joblib


""


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
0,2026-05-26,100000.0,100000.0,0,-0.5,-0.5,0.200764,FORCE_BUY,0.24,0.26


In [1]:
from importlib import reload
import adaptive_reward_checkpoint_fresh
reload(adaptive_reward_checkpoint_fresh)

from adaptive_reward_checkpoint_fresh import run_adaptive_reward_yfinance_intraday_loop

live_res = run_adaptive_reward_yfinance_intraday_loop(
    checkpoint_path="output_adaptive_reward_resumed_fresh_TQQQ_252days_at_2020/final_checkpoint.joblib",
    output_dir="output_adaptive_reward_live_TQQQ",
    initial_capital=100000.0,
    fee_pct=0.0,
    save_snapshot_path="output_adaptive_reward_live_TQQQ/live_checkpoint.joblib",
    poll_seconds=60,
    market_close_time="16:00",
    timezone="America/New_York",
    verbose=True,
)

print("latest checkpoint:", live_res["latest_checkpoint_path"])
display(live_res["trades_df"].tail())
display(live_res["daily_log_df"].tail())

[LIVE] advancing through 2026-05-29 16:00:00 EDT
[TRAIN][5M] asof=2026-05-26 feats=69 buy=YES sell=YES rows=121233
[SAVED] output_adaptive_reward_live_TQQQ\trades.csv
[SAVED] output_adaptive_reward_live_TQQQ\daily_log.csv
[SAVED] output_adaptive_reward_live_TQQQ\daily_reward_log.csv
[SAVED] output_adaptive_reward_live_TQQQ\equity_vs_buyhold.png
[SAVED] output_adaptive_reward_live_TQQQ\price_with_trades.png
[SAVED] output_adaptive_reward_live_TQQQ\p_day.png
[CHECKPOINT] saved continued snapshot -> output_adaptive_reward_live_TQQQ/live_checkpoint.joblib
[LIVE] checkpoint -> output_adaptive_reward_live_TQQQ/live_checkpoint.joblib
[LIVE STATUS]
  today's decision: date=2026-05-29 00:00:00 daily_action=FORCE_BUY p_day=0.22061903632875482 buy_level=0.24 sell_level=0.26
  account: date=2026-05-29 00:00:00 equity=100000.0 cash=100000.0 pos=0
  5m model: buy_th=-0.5 sell_th=-0.5
  last trading decision: None
[LIVE] reached market close 2026-05-29 16:00:00 EDT
latest checkpoint: output_adaptive_

""


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
0,2026-05-26,100000.0,100000.0,0,-0.5,-0.5,0.200764,FORCE_BUY,0.24,0.26
1,2026-05-27,100000.0,100000.0,0,-0.5,-0.5,0.215126,FORCE_BUY,0.24,0.26
2,2026-05-28,100000.0,100000.0,0,-0.5,-0.5,0.209231,FORCE_BUY,0.24,0.26
3,2026-05-29,100000.0,100000.0,0,-0.5,-0.5,0.220619,FORCE_BUY,0.24,0.26
